# Cyclospora outbreak

In [1]:
import requests
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
import pandas as pd
import re

In [2]:
playwright = await async_playwright().start()
browser = await playwright.chromium.launch(headless=True)

In [3]:
page = await browser.new_page()

In [4]:
await page.goto("https://www.cdc.gov/cyclosporiasis/php/surveillance/index.html")

<Response url='https://www.cdc.gov/cyclosporiasis/php/surveillance/index.html' request=<Request url='https://www.cdc.gov/cyclosporiasis/php/surveillance/index.html' method='GET'>>

In [6]:
async with page.expect_download() as download_info:
    await page.click('a[aria-label="Download this data in a CSV file format."]')

download = await download_info.value
await download.save_as("data/data.csv")
print("Downloaded to:", await download.path())

Downloaded to: /var/folders/7_/ltwct_mx5q5c5xzx5gnl5p3m0000gr/T/playwright-artifacts-ZHGXaa/4fdc7e3b-3887-4b98-85d0-4b8f8ac91197


In [7]:
df = pd.read_csv("data/data.csv")

In [9]:
df.head()

,Location,Number of Sick People
0,Alaska,1 to 10
1,Arkansas,1 to 10
2,California,1 to 10
3,Colorado,1 to 10
4,Connecticut,1 to 10


In [11]:
df['min'] = df['Number of Sick People'].str.extract(r'(\d+)\s+to\s+\d+').astype(int)
df['max'] = df['Number of Sick People'].str.extract(r'\d+\s+to\s+(\d+)').astype(int)

In [12]:
df['average'] = df[['min', 'max']].mean(axis=1)

In [17]:
df.sort_values(by='average', ascending=False, inplace=True)
df.head()

,Location,Number of Sick People,min,max,average
15,Michigan,161 to 300,161,300,230.5
20,New York,81 to 160,81,160,120.5
7,Illinois,31 to 80,31,80,55.5
21,North Carolina,31 to 80,31,80,55.5
11,Kentucky,31 to 80,31,80,55.5


In [19]:
df.to_csv("data/clean_data.csv", index=False)